# Lab 6.1 &mdash; Your First Tool with @tool

**Level:** Beginner &nbsp;|&nbsp; **Est. time:** 20 min &nbsp;|&nbsp; **Day 3 &middot; Module 6 &mdash; Frameworks for Building AI Agents**

### What you'll do
- Wrap a function as a tool with LangChain's @tool decorator
- See the name, description (from the docstring) and args the model reads
- Bind your tools to a real agent and watch the model call them

> **How this lab works (near-real):** you have a local Ollama running `llama3.1:8b`. Read the **Concept**, fill the real `___` blanks in **Build it** (real tool bodies, real prompts, the real `create_agent` call), then **Run it for real** &mdash; a real LLM drives a real agent over real tools &mdash; and **read the trace** to see exactly what it did. Finish with an open **Your turn**. There is **no auto-grader**; the goal is a working agent and a trace you can read.

> **Framework note:** these labs use the **real** LangChain (`langchain`, `langchain-core`, `langchain-ollama`, `langgraph`) and a **real local model** (`ollama run llama3.1:8b`, pinned to `http://127.0.0.1:11434`). If Ollama is down, the run cells print how to start it instead of crashing. A `@tool` must **catch its own errors and return a string** &mdash; a tool that *raises* aborts the whole agent run (you'll see exactly this in Lab 11).

**Reference:** [Module 6 slides &mdash; Defining a tool](../../presentation/day3-module6-frameworks-for-building-ai-agents.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 6 labs](./index.html)

In [ ]:
# Setup -- run me first
import os, socket, pathlib
from dotenv import load_dotenv
load_dotenv(pathlib.Path("/home/rajesh/Training/courses/building-intelligents-ai-agents/.env"))   # SERPER_API_KEY, WOLFRAM_ALPHA_APPID, GROQ/OPENAI keys

WORK = "/tmp/biaa-lab-06-01"
os.makedirs(WORK, exist_ok=True)

def ollama_up(host="127.0.0.1", port=11434):
    """True if a local Ollama server is listening. If it's down, start it with:  ollama run llama3.1:8b"""
    try:
        with socket.create_connection((host, port), timeout=1):
            return True
    except OSError:
        return False

from langchain_ollama import ChatOllama
# Day-3 provider: a REAL local model. Pin the host -- plain 'localhost' can give 'No route to host'.
llm = ChatOllama(model="llama3.1:8b", temperature=0, base_url="http://127.0.0.1:11434")

def print_trace(result):
    """Print a REAL agent message trace: tool calls the model made, tool observations, final answer."""
    for m in result["messages"]:
        for tc in (getattr(m, "tool_calls", None) or []):
            print("TOOL CALL:", tc["name"], tc["args"])
        if type(m).__name__ == "ToolMessage":
            print("OBS:", str(m.content)[:200])
        elif str(getattr(m, "content", "")).strip():
            print(type(m).__name__, ":", str(m.content)[:300])

if ollama_up():
    print("Ollama reachable at 127.0.0.1:11434 | model:", llm.model, "| WORK:", WORK)
else:
    print("Ollama NOT reachable -- start it with:  ollama run llama3.1:8b")
    print("(The 'Run it for real' cells will print this note instead of crashing.)  WORK:", WORK)

## Concept
In Module 5 you built tools by hand as dicts. A framework does it for you: LangChain's **`@tool`**
decorator (from **`langchain_core.tools`**) turns a plain function into a **`StructuredTool`** with a
**name**, a **description** (taken from the **docstring** &mdash; the text the model reads to decide when
to use it), an **args** schema, and an **`.invoke()`** method. Notice the calculator **catches its own
error and returns a string** &mdash; never let a tool raise.

In [ ]:
from langchain_core.tools import tool

@tool
def greet(name: str) -> str:
    """Say hello to someone by name."""
    return "Hello, " + name + "!"

print("name:", greet.name, "| description:", greet.description)
print("args:", list(greet.args))
print("greet.invoke('Ada') ->", greet.invoke("Ada"))

## Build it
Write two **real** tools &mdash; a **calculator** (safe arithmetic, wrapped so it never raises) and a
**lookup** &mdash; then call them with `.invoke()`.

In [ ]:
import ast, operator
# safe arithmetic: walk a parsed AST of numbers + (+ - * / ** unary-minus) -- never bare eval()
_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
        ast.Div: operator.truediv, ast.USub: operator.neg, ast.Pow: operator.pow}
def safe_calc(expr):
    def ev(node):
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return node.value
        if isinstance(node, ast.BinOp):
            return _OPS[type(node.op)](ev(node.left), ev(node.right))
        if isinstance(node, ast.UnaryOp):
            return _OPS[type(node.op)](ev(node.operand))
        raise ValueError("unsupported expression")
    return ev(ast.parse(expr, mode="eval").body)

from langchain_core.tools import tool

@tool
def calculator(expression: str) -> str:
    """___  (TODO: one line telling the model WHEN to use this -- it is the tool's API)."""
    try:
        return str(safe_calc(expression))          # success path
    except Exception:
        return "error: not a numeric expression"    # a tool must NEVER raise -- return a string

@tool
def lookup(key: str) -> str:
    """Look up a known fact by its key, for example 'capital of france'."""
    facts = {"capital of france": "Paris", "population of metropolis": "120000"}
    return ___    # TODO: return the fact for key (lowercased/stripped), else "unknown"

try:
    print("calculator.name :", calculator.name)
    print("calculator.args :", list(calculator.args))
    print("calculator.invoke('120000/2') =", calculator.invoke("120000/2"))
    print("calculator.invoke('oops')     =", calculator.invoke("oops"))   # returns a string, no crash
    print("lookup.invoke('capital of france') =", lookup.invoke("capital of france"))
except Exception as e:
    print("(Fill the ___ blanks above, then re-run.)", type(e).__name__)

## Run it for real &amp; read the trace
Bind both of YOUR tools to a real agent and ask a question that needs each one. Read the trace: the model calls your functions.

_This calls the real `llama3.1:8b` via Ollama. If Ollama is down the cell prints how to start it instead of crashing._

In [ ]:
if not ollama_up():
    print("Ollama not reachable -- start `ollama run llama3.1:8b`, then re-run this cell.")
else:
    try:
        from langchain.agents import create_agent
        agent = create_agent(llm, tools=[calculator, lookup])
        result = agent.invoke({"messages": [("user",
                 "What is the capital of France, and what is 120000 divided by 2?")]},
                 config={"recursion_limit": 8})
        print_trace(result)
    except Exception as e:
        print("(Fill the ___ blanks above, then re-run.)", type(e).__name__)

## What to notice
- The trace shows **`TOOL CALL: lookup {...}`** and **`TOOL CALL: calculator {...}`** &mdash; the model chose to call *your* Python functions.
- Each **`OBS:`** line is what your tool returned; the model reads it and continues.
- A small model can still make a wrong call now and then &mdash; that's real agent behaviour, and it's why we read traces.

## Your turn (open task &mdash; no grader)
Add a **third** tool of your own &mdash; e.g. a `word_count(text)` tool with a clear docstring &mdash; put it
in the `tools=[...]` list, and re-run the agent with a question that needs it.
**What good looks like:** the trace shows your new tool being called with sensible args, and the final
answer uses its result. If the model ignores your tool, sharpen the docstring (that's the tool's real API).

---
### Key takeaway
`@tool` turns a function into a real Tool the agent can call, and the docstring is the API the model reads. You just watched llama3.1:8b call your own Python -- next we make descriptions drive which tool it picks.

[&#8592; All Module 6 labs](./index.html) &nbsp;&middot;&nbsp; [Module 6 slides](../../presentation/day3-module6-frameworks-for-building-ai-agents.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>